In [0]:
%pip install kaggle

In [0]:
%restart_python

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS real_estate;

CREATE SCHEMA IF NOT EXISTS real_estate.raw;

CREATE VOLUME IF NOT EXISTS real_estate.raw.raw_data;

In [0]:
#setting up kaggle config path
import os

os.environ['KAGGLE_CONFIG_DIR'] = '/Volumes/real_estate/raw/raw_data/'

In [0]:
#Downloading dataset from kaggle
!kaggle datasets download -d kanchana1990/washington-real-estate-sold-properties-data-2026 \
-p /Volumes/real_estate/raw/raw_data/ \
--unzip

In [0]:
dbutils.fs.ls("dbfs:/Volumes/real_estate/raw/raw_data/")

In [0]:
#loading data into spark
from pyspark.sql.types import StructType, StructField, StringType, FloatType

schema = StructType([
    StructField("zip", StringType(), True),
    StructField("type", StringType(), True),
    StructField("year_built", FloatType(), True),
    StructField("listPrice", FloatType(), True),
    StructField("lastSoldPrice", FloatType(), True),
    StructField("list_to_sold_ratio", FloatType(), True),
    StructField("sqft", FloatType(), True),
    StructField("price_per_sqft", FloatType(), True),
    StructField("stories", FloatType(), True),
    StructField("beds", FloatType(), True),
    StructField("baths", FloatType(), True),
    StructField("baths_full", FloatType(), True),
    StructField("baths_full_calc", FloatType(), True),
    StructField("garage", FloatType(), True),
    StructField("sanitized_text", StringType(), True)
])

file_path = '/Volumes/real_estate/raw/raw_data/washington_ultimate.csv'

df = spark.read.csv(
    file_path,
    header=True,
    schema=schema
)

display(df, limit=10)

In [0]:
df.printSchema()

In [0]:
#adding a primary key to the table
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn('property_id', monotonically_increasing_id())

display(df, limit=10)

In [0]:
print(f'Row count: {df.count()}')

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS real_estate.bronze;

In [0]:
df.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('real_estate.bronze.washington_ultimate')

In [0]:
display(spark.sql('select * from real_state.bronze.washington_ultimate limit 10'))